# MIE312 Lab 3 - External Flow: Lift and Drag on Airfoils
Developed by Prof. Sullivan (pierre.sullivan@utoronto.ca). Suggestions for improvements are welcome.

This Colab-ready notebook follows the Fall 2026 Lab 3 manual. It converts the raw GUNT force-channel readings to the physical lift sign convention, corrects measured drag for the holder, calculates Reynolds number and aerodynamic coefficients, estimates a stall-onset range from the $C_L$ curve, and performs the requested 0° and 90° comparison calculations using the velocity measured at each corresponding angle.

## How to use in Colab
1. Open this notebook from the link in the lab manual and choose **File → Save a copy in Drive**.
2. Run the setup cells.
3. Replace each `np.nan` in the measurement table with your readings, and enter the velocity measured at each angle.
4. Run the analysis sections in order.
5. Run Section L to download the figures and tables for your report.

The handout measurement table uses angles 0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 40, 50, 60, 70, 80, and 90 degrees.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.precision', 4)
from IPython.display import display

# Folder for the figures and tables saved for your report (emptied each time this cell runs)
import shutil
from pathlib import Path
OUT_DIR = Path("lab3_outputs")
shutil.rmtree(OUT_DIR, ignore_errors=True)
OUT_DIR.mkdir()

def _has_finite_data(fig):
    """True if any line, marker or bar in the figure has a finite value."""
    for ax in fig.axes:
        for line in ax.get_lines():
            if np.isfinite(np.asarray(line.get_ydata(), float)).any():
                return True
        for coll in ax.collections:
            offsets = np.asarray(coll.get_offsets(), float)
            if offsets.size and np.isfinite(offsets).all(axis=1).any():
                return True
        for patch in ax.patches:
            if np.isfinite(patch.get_height()) and patch.get_height() != 0:
                return True
    return False

def save_fig(name):
    """Save the current figure as a 300-dpi PNG for your report (called just before plt.show())."""
    fig = plt.gcf()
    if not _has_finite_data(fig):
        return  # nothing plotted yet, so don't save an empty figure
    if globals().get("USE_EXAMPLE_DATA", False):
        fig.text(0.5, 0.5, "EXAMPLE DATA", fontsize=48, color="red", alpha=0.2,
                 ha="center", va="center", rotation=30, transform=fig.transFigure)
    fig.savefig(OUT_DIR / f"{name}.png", dpi=300, bbox_inches="tight")

## Example data (optional)
The measurement table in this notebook starts blank: replace each `np.nan` with your own reading. The example is the dataset from the supplied sample report; its lift-channel values use the instrument sign convention, so they are entered as raw $F_y$ readings.

To see how the notebook works before you have data, set `USE_EXAMPLE_DATA = True` below and choose **Runtime → Run all**. The example values then replace your tables, every saved figure is stamped **EXAMPLE DATA**, and the summary is flagged. **Set it back to `False` before you analyze your own measurements.**

In [ ]:
USE_EXAMPLE_DATA = False

# Example measurements from the supplied sample report (used only when USE_EXAMPLE_DATA = True)
EXAMPLE_DATA = {
    "airfoil": pd.DataFrame({
        'angle_deg': [0.3509, 3.5054, 6.3158, 9.1139, 12.6193, 15.7741, 18.9289, 22.4343, 24.5375, 27.6923, 30.8471, 40.3116, 51.1782, 60.6426, 71.0895, 80.6232, 90.7887],
        'Fy_raw_N': [0.0684, -0.1172, -0.3027, -0.4785, -0.6348, -0.7910, -0.9277, -0.7031, -0.7715, -0.8984, -0.9961, -1.0937, -1.0254, -0.9082, -0.6738, -0.4199, -0.0684],
        'drag_measured_N': [0.0684, 0.0586, 0.0586, 0.1172, 0.1562, 0.1855, 0.2441, 0.4102, 0.4980, 0.5957, 0.7031, 1.0254, 1.3086, 1.6602, 1.9336, 2.0996, 2.0996],
        'velocity_m_s': [15.0]*17,
    }),
}

EXAMPLE_WARNING = (
    "⚠️ EXAMPLE DATA: these results use the built-in example values, not your measurements. "
    "Set USE_EXAMPLE_DATA = False and enter your own data before using anything in your report."
)

def use_example_if_requested(df, key, measured_cols):
    """Return the example table when USE_EXAMPLE_DATA is True; otherwise check the table you entered."""
    if USE_EXAMPLE_DATA:
        print(EXAMPLE_WARNING)
        return EXAMPLE_DATA[key].copy()
    if df[measured_cols].isna().all().all():
        print("No measurements entered yet: replace the np.nan entries with your readings.")
    else:
        ex = EXAMPLE_DATA[key]
        if df.shape == ex.shape and np.allclose(df[measured_cols].to_numpy(float),
                                                ex[measured_cols].to_numpy(float), equal_nan=True):
            print("⚠️ These values are identical to the example data. Enter your own measurements.")
    return df

## Section A — Enter your measured data
Enter the measured angle of attack, **raw GUNT $F_y$ lift-channel reading**, measured drag-channel reading, and fan velocity.

With the standard axis setup in the lab, upward physical lift appears as a negative raw $F_y$ value. The notebook therefore uses

$$F_L=-F_y$$

before calculating $C_L$, $L/D$, or lift maxima. If your TA confirms that the software axis was configured differently, change the sign-conversion line in the data-reduction cell and document the convention in your report.

In [ ]:
# Enter your measurements: replace each np.nan with your reading
data = pd.DataFrame({
    'angle_deg': [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 40, 50, 60, 70, 80, 90],
    'Fy_raw_N': [np.nan]*17,
    'drag_measured_N': [np.nan]*17,
    'velocity_m_s': [np.nan]*17,
})
data = use_example_if_requested(data, "airfoil", ["Fy_raw_N", "drag_measured_N", "velocity_m_s"])
data

## Section B — Constants and geometry
The handout gives the airfoil chord as 0.1 m, thickness 15 mm, airfoil reference area 0.01 m$^2$, holder diameter 4 mm, holder length 0.31 m, and holder aerodynamic resistance parameter $C_DA_{front}=0.000125\,m^2$.

In [ ]:
# Air properties: dry air at 20 deg C and 1 atm
rho_air = 1.204         # kg/m^3
mu_air = 1.81e-5        # Pa*s

# Airfoil and holder geometry
A_airfoil = 0.01        # m^2
c_airfoil = 0.10        # m
t_airfoil = 0.015       # m
CDA_holder = 0.000125   # m^2

print('Assumed air properties: dry air at 20 deg C and 1 atm')
print(f'rho = {rho_air:.3f} kg/m^3, mu = {mu_air:.3e} Pa*s')
rho_air, mu_air, A_airfoil, c_airfoil, t_airfoil, CDA_holder

## Section C — Force sign convention and holder-drag correction
For the standard GUNT setup,

$$F_L=-F_y.$$

The drag measured by the force sensor includes the holder contribution. The corrected airfoil drag is

$$F_{D,airfoil}=F_{D,measured}-F_{D,holder},$$

with

$$F_{D,holder}=\tfrac12\rho V^2(C_DA)_{front}.$$

Because velocity may vary slightly from point to point, the holder correction is calculated using the measured velocity on each row.

In [ ]:
data = data.sort_values('angle_deg').reset_index(drop=True).copy()

# Standard GUNT sign convention: upward physical lift is negative raw Fy.
data['lift_N'] = -data['Fy_raw_N']
data['holder_drag_N'] = 0.5 * rho_air * data['velocity_m_s']**2 * CDA_holder
data['drag_corrected_N'] = data['drag_measured_N'] - data['holder_drag_N']

if (data['drag_corrected_N'].dropna() <= 0).any():
    print('NOTE: One or more corrected drag values are non-positive. Check taring, sign convention, and holder correction before interpreting C_D or L/D at those points.')

data[['angle_deg', 'Fy_raw_N', 'lift_N', 'drag_measured_N', 'holder_drag_N', 'drag_corrected_N']]

## Section D — Lift and corrected drag vs angle of attack
Plot the **physical lift** $F_L=-F_y$ and the holder-corrected airfoil drag. These are the force quantities used for the aerodynamic interpretation.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(data['angle_deg'], data['lift_N'], marker='o', label='Physical lift $F_L$')
plt.plot(data['angle_deg'], data['drag_corrected_N'], marker='s', label='Corrected airfoil drag')
plt.xlabel('Angle of attack (deg)')
plt.ylabel('Force (N)')
plt.title('Lift and corrected drag vs angle of attack')
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("D_lift_and_drag_vs_angle")
plt.show()

## Section E — Stall-onset estimate from $C_L$
The stall estimate is based on the **lift coefficient** rather than the raw lift force. To avoid identifying a single noisy point as stall, the notebook finds the first pre-stall local maximum in $C_L$ for which the **next two measured $C_L$ values both remain below that maximum**.

Because the measurements are discrete, the notebook reports an **approximate stall-onset range** extending from the candidate $C_L$ maximum to the next measured angle. This range should be checked against the plotted data and any observations of separation, vibration, or unsteadiness made during the experiment.

In [ ]:
angles = data['angle_deg'].to_numpy(dtype=float)
lift = data['lift_N'].to_numpy(dtype=float)
velocity = data['velocity_m_s'].to_numpy(dtype=float)
q_local = 0.5 * rho_air * velocity**2
cl_for_stall = lift / (q_local * A_airfoil)

stall_idx = None
for i in range(1, len(cl_for_stall) - 2):
    vals = cl_for_stall[i-1:i+3]
    if np.all(np.isfinite(vals)):
        is_local_peak = cl_for_stall[i] >= cl_for_stall[i-1]
        sustained_below = (cl_for_stall[i+1] < cl_for_stall[i]) and (cl_for_stall[i+2] < cl_for_stall[i])
        if is_local_peak and sustained_below:
            stall_idx = i
            break

stall_peak_angle = angles[stall_idx] if stall_idx is not None else np.nan
stall_peak_CL = cl_for_stall[stall_idx] if stall_idx is not None else np.nan
stall_range_upper = angles[stall_idx + 1] if stall_idx is not None else np.nan

if np.isfinite(cl_for_stall).any():
    full_max_idx = int(np.nanargmax(cl_for_stall))
    full_max_CL_angle = angles[full_max_idx]
    full_max_CL = cl_for_stall[full_max_idx]
else:
    full_max_idx = None
    full_max_CL_angle = np.nan
    full_max_CL = np.nan

if np.isfinite(stall_peak_angle):
    print(f'Pre-stall C_L maximum: {stall_peak_CL:.4f} at {stall_peak_angle:.3f} deg')
    print(f'Estimated stall-onset range: {stall_peak_angle:.3f} to {stall_range_upper:.3f} deg')
    print('Criterion: the next two measured C_L values remain below the candidate maximum.')
else:
    print('Stall-onset estimate: not available - enter complete data and verify the C_L plot manually.')

if np.isfinite(full_max_CL_angle):
    print(f'Largest C_L over the full sweep: {full_max_CL:.4f} at {full_max_CL_angle:.3f} deg')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(data['angle_deg'], cl_for_stall, marker='o', label='$C_L$')
if np.isfinite(stall_peak_angle):
    plt.axvspan(stall_peak_angle, stall_range_upper, alpha=0.15,
                label=f'Estimated stall onset: {stall_peak_angle:.1f}°-{stall_range_upper:.1f}°')
    plt.axvline(stall_peak_angle, linestyle='--', label=f'Pre-stall $C_L$ max ≈ {stall_peak_angle:.1f}°')
if np.isfinite(full_max_CL_angle) and (not np.isfinite(stall_peak_angle) or full_max_idx != stall_idx):
    plt.axvline(full_max_CL_angle, linestyle=':', label=f'Full-sweep max $C_L$ ≈ {full_max_CL_angle:.1f}°')
plt.xlabel('Angle of attack (deg)')
plt.ylabel('$C_L$')
plt.title('Lift coefficient and estimated stall-onset range')
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("E_CL_stall_onset")
plt.show()

## Section F — Reynolds number
Question 5 asks for the airfoil Reynolds number. The handout gives the needed chord length and the report evaluates $Re=\rho U c/\mu$ using $c=0.1\,m$ and $U=15\,m/s$.

In [ ]:
data['Re'] = rho_air * data['velocity_m_s'] * c_airfoil / mu_air
data[['angle_deg', 'velocity_m_s', 'Re']].head()


## Section G — Lift and drag coefficients
Using the airfoil planform reference area $A=0.01\,\mathrm{m^2}$,

$$C_L=\frac{F_L}{\tfrac12\rho V^2A},\qquad C_D=\frac{F_{D,airfoil}}{\tfrac12\rho V^2A}.$$

The lift-to-drag ratio is evaluated only where the corrected drag is positive.

In [ ]:
q = 0.5 * rho_air * data['velocity_m_s']**2
data['CL'] = data['lift_N'] / (q * A_airfoil)
data['CD'] = data['drag_corrected_N'] / (q * A_airfoil)
data['L_over_D'] = np.where(data['drag_corrected_N'] > 0, data['lift_N'] / data['drag_corrected_N'], np.nan)

coef_table = data[['angle_deg', 'lift_N', 'drag_corrected_N', 'CL', 'CD', 'L_over_D', 'Re']]
coef_table

## Section H — Optimal angle of attack
For this lab, define the optimal measured angle as the angle giving the maximum finite $L/D$. This need not coincide with either the stall-associated pre-stall lift maximum or the largest force observed in the post-stall sweep.

In [ ]:
ld = data['L_over_D'].to_numpy(dtype=float)
if np.isfinite(ld).any():
    optimal_idx = int(np.nanargmax(ld))
    optimal_angle = float(data.loc[optimal_idx, 'angle_deg'])
    optimal_LD = float(data.loc[optimal_idx, 'L_over_D'])
    print(f'Optimal measured angle of attack (max L/D): {optimal_angle:.3f} deg')
    print(f'Maximum L/D ratio: {optimal_LD:.4f}')
else:
    optimal_idx = None
    optimal_angle = np.nan
    optimal_LD = np.nan
    print('Optimal angle: not available - enter complete force data first.')

if np.isfinite(stall_peak_angle):
    print(f'Estimated stall-onset range: {stall_peak_angle:.3f} to {stall_range_upper:.3f} deg')
if np.isfinite(full_max_CL_angle):
    print(f'Full-sweep maximum-C_L angle: {full_max_CL_angle:.3f} deg')

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(data['angle_deg'], data['L_over_D'], marker='o')
if np.isfinite(optimal_angle):
    plt.axvline(optimal_angle, linestyle='--', label=f'Optimal angle ≈ {optimal_angle:.1f}°')
plt.xlabel('Angle of attack (deg)')
plt.ylabel('L/D ratio')
plt.title('Aerodynamic efficiency vs angle of attack')
plt.grid(True, alpha=0.3)
if np.isfinite(optimal_angle):
    plt.legend()
save_fig("H_L_over_D")
plt.show()

## Section I — Simple drag comparisons at 0° and 90°
At 0°, use an aligned laminar flat-plate skin-friction comparison with the **airfoil chord $c=0.10$ m as the streamwise plate length**, span $b=0.10$ m, and both wetted sides. The 15-mm airfoil thickness is retained as a real-geometry parameter but is not the Blasius streamwise length.

At 90°, the lab adopts an idealized **thin plate normal to the flow** with projected area $A_{90}=0.010$ m$^2$ and $C_D=1.9$ (Hoerner, *Fluid-Dynamic Drag*, 1965). This is a prescribed bluff-body comparison, not an exact prediction for the finite-span airfoil.

Each estimate uses the **measured velocity at its corresponding angle**: the nearest measured point to 0° for the aligned-plate estimate and the nearest measured point to 90° for the normal-plate estimate.

In [ ]:
row_0 = data.iloc[(data['angle_deg'] - 0).abs().argsort()[:1]]
row_90 = data.iloc[(data['angle_deg'] - 90).abs().argsort()[:1]]

U_0 = float(row_0['velocity_m_s'].iloc[0])
U_90 = float(row_90['velocity_m_s'].iloc[0])

b_airfoil = A_airfoil / c_airfoil  # 0.10 m span
L_flat_0 = c_airfoil               # streamwise Blasius length

# 0 deg: total laminar skin-friction drag on both wetted sides of an aligned flat plate.
D_flat_0 = 2 * 0.664 * b_airfoil * (rho_air**0.5) * (mu_air**0.5) * (U_0**1.5) * (L_flat_0**0.5)

# 90 deg: prescribed idealized thin plate normal to flow (Hoerner, 1965).
CD_flat_90 = 1.9
A_flat_90 = c_airfoil * b_airfoil
D_flat_90 = 0.5 * rho_air * U_90**2 * CD_flat_90 * A_flat_90

comparison = pd.DataFrame({
    'case': ['0 deg aligned flat-plate estimate', '90 deg normal-plate estimate'],
    'velocity_used_m_s': [U_0, U_90],
    'theoretical_drag_N': [D_flat_0, D_flat_90],
    'nearest_experimental_angle_deg': [row_0['angle_deg'].values[0], row_90['angle_deg'].values[0]],
    'experimental_corrected_drag_N': [row_0['drag_corrected_N'].values[0], row_90['drag_corrected_N'].values[0]],
})
comparison['percent_difference_%'] = 100 * (comparison['experimental_corrected_drag_N'] - comparison['theoretical_drag_N']) / comparison['theoretical_drag_N']
comparison

## Section J — Summary table for report use
This final table is usually the most useful one to copy into a report appendix or results section.

In [ ]:
if USE_EXAMPLE_DATA:
    print(EXAMPLE_WARNING)

summary_cols = ['angle_deg', 'Fy_raw_N', 'lift_N', 'drag_measured_N', 'holder_drag_N', 'drag_corrected_N', 'CL', 'CD', 'L_over_D', 'Re']
summary_table = data[summary_cols].copy()
summary_table

## Section K — Short discussion helper
Use these points to shape your written discussion:
- Lift usually rises with angle at low incidence, while drag grows more slowly at first.
- The optimal angle is where $L/D$ is highest, not necessarily where lift is largest.
- Stall corresponds to the onset of flow separation and a loss of lift growth.
- At 90°, pressure drag dominates and the airfoil behaves more like a bluff body.
- Surface roughness tends to matter more at low angles through boundary-layer behavior, while at 90° pressure drag dominates the picture.
- Which measurements limit the accuracy of $C_L$ and $C_D$ most: the fluctuating force readings, the velocity, or the angle setting? How do these uncertainties affect your stall-angle estimate?

These are the same interpretive themes emphasized by the handout and reflected in the sample report discussion.

## Section L — Save figures and tables for your report
Run this cell after the analysis. It saves every figure above as a 300-dpi PNG and every results table as a CSV file, then downloads them together as `lab3_outputs.zip`. If your browser blocks the download, open the **Files** panel on the left and download the zip from there.

In [ ]:
# Results tables to save -- file name: (results table, the measurement table it comes from)
TABLES = {
    "results_all_angles": ("data", "data"),
    "coefficients": ("coef_table", "data"),
    "drag_comparison_0_90": ("comparison", "data"),
    "summary": ("summary_table", "data")
}

saved_tables = []
def has_measurements(df):
    """True if any measurement column (all columns after the first) holds a number."""
    return isinstance(df, pd.DataFrame) and df.iloc[:, 1:].select_dtypes("number").notna().any().any()

for name, (var, source) in TABLES.items():
    table = globals().get(var)
    if isinstance(table, pd.DataFrame) and has_measurements(globals().get(source)):
        table.to_csv(OUT_DIR / f"{name}.csv", index=False)
        saved_tables.append(f"{name}.csv")

saved_figures = sorted(p.name for p in OUT_DIR.glob("*.png"))
if globals().get("USE_EXAMPLE_DATA", False):
    print(EXAMPLE_WARNING)
print("Figures:", ", ".join(saved_figures) if saved_figures else "none yet (run the plotting cells first)")
print("Tables: ", ", ".join(saved_tables) if saved_tables else "none yet (enter your measurements first)")

zip_path = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Not running in Colab; outputs saved to {zip_path}")

## Section M — Submission checklist

Before submitting, consider adding:
- your raw data table from the lab,
- captions under each figure,
- a statement of the lift sign convention ($F_L=-F_y$) and the holder-drag correction you applied,
- one short uncertainty paragraph,
- one short conclusion paragraph,
- any TA-specific formatting requirements.

This notebook is designed to save time on repetitive calculations so you can focus on interpretation and discussion. You are still responsible for understanding the equations, checking the inputs, and showing any worked example required by the lab instructions.